### Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset
from torch.optim import Optimizer
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import wandb
import random
import os

In [ ]:
torch.manual_seed(42)

In [ ]:
PROJECT_NAME = "recursive-synaptic-balance"

default_config = {
    "epochs": 4,
    "optimizer": "sgd",
    "loss_type": "rsb",
    "dataset": "CIFAR10",
    "batch_size": 1, # Keep at 1
    "lr": 0.01,
    "alpha": 0.01,
    "beta": 0.01,
    "model_type": "CNN3",
    "val_fraction": 0.1,
}

sweep_config = {
    "name": "rsb_cnn_cifar10",
    "method": "bayes",
    "metric": {
        "name": "val_accuracy",
        "goal": "maximize",
    },
    "run_cap": 30,
    "parameters": {
        "lr": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },
        "alpha": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },
        "beta": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },
    },
    "early_terminate": {
        "type": "hyperband",
        "min_iter": 1,
        "max_iter": 30,
    },
}

### Datasets

In [ ]:
mnist_transform = transforms.Compose([ # It would be nice to check these numbers, if we have time
    transforms.ToTensor(), 
    transforms.Normalize((0.1307,), (0.3081,))
])
cifar10_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])
cifar100_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

In [ ]:
def get_data_loaders(dataset_type, batch_size, val_fraction=0.1, test_fraction=0.2, shuffle=True, pin_memory=True):
    if dataset_type == "MNIST":
        dataset_class = datasets.MNIST
        transform = mnist_transform
    elif dataset_type == "CIFAR10":
        dataset_class = datasets.CIFAR10
        transform = cifar10_transform
    elif dataset_type == "CIFAR100":
        dataset_class = datasets.CIFAR100
        transform = cifar100_transform
    else:
        raise ValueError(f"'{dataset_type}' not defined as a dataset.")

    base_train = dataset_class("./data", train=True, download=True, transform=transform)
    base_test  = dataset_class("./data", train=False, download=True, transform=transform)

    full_dataset   = ConcatDataset([base_train, base_test])
    total_size     = len(full_dataset)
    test_size      = int(test_fraction * total_size)
    train_val_size = total_size - test_size

    generator = torch.Generator().manual_seed(42)
    train_val_dataset, test_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_val_size, test_size],
        generator=generator,
    )

    val_size = int(val_fraction * train_val_size)
    train_size = train_val_size - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        train_val_dataset,
        [train_size, val_size],
        generator=generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader, test_loader


### Models

In [ ]:
class CNN1(nn.Module):
    def __init__(self, input_size, in_channels=1, num_classes=10):
        super().__init__()
        self.maxpool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        h, w = input_size
        self.fc = nn.Linear(64 * (h // 4) * (w // 4), num_classes)

    def forward(self, x):
        x = self.maxpool(self.relu(self.conv1(x)))
        x = self.maxpool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return F.log_softmax(x, dim=1)


class CNN2(nn.Module):
    def __init__(self, input_size, in_channels=1, num_classes=10):
        super().__init__()
        self.maxpool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        h, w = input_size
        self.fc1 = nn.Linear(128 * (h // 4) * (w // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.maxpool(self.relu(self.conv1(x)))
        x = self.maxpool(self.relu(self.conv2(x)))
        x = self.relu(self.conv3(x))

        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


class CNN3(nn.Module):
    def __init__(self, input_size, in_channels=1, num_classes=10):
        super().__init__()
        self.maxpool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        h, w = input_size
        self.fc1 = nn.Linear(128 * (h // 4) * (w // 4), 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.maxpool(self.relu(self.conv1(x)))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.maxpool(self.relu(self.conv4(x)))

        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [ ]:
class FNN1(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


class FNN2(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


class FNN3(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

In [ ]:
def get_model(config, device):
    model_type = getattr(config, "model_type")
    dataset = config.dataset

    if dataset == "MNIST":
        input_size = (28, 28)
        in_channels = 1
        input_dim = 28 * 28
        output_dim = 10
    elif dataset == "CIFAR10":
        input_size = (32, 32)
        in_channels = 3
        input_dim = 32 * 32 * 3
        output_dim = 10
    elif dataset == "CIFAR100":
        input_size = (32, 32)
        in_channels = 3
        input_dim = 32 * 32 * 3
        output_dim = 100
    else:
        raise ValueError(f"'{dataset}' not supported as a dataset.")

    if model_type in {"FNN1", "FNN2", "FNN3"}:
        if model_type == "FNN1":
            model = FNN1(input_dim=input_dim, output_dim=output_dim)
        elif model_type == "FNN2":
            model = FNN2(input_dim=input_dim, output_dim=output_dim)
        else:  # "FNN3"
            model = FNN3(input_dim=input_dim, output_dim=output_dim)

    elif model_type in {"CNN1", "CNN2", "CNN3"}:
        if model_type == "CNN1":
            model = CNN1(input_size=input_size, in_channels=in_channels, num_classes=output_dim)
        elif model_type == "CNN2":
            model = CNN2(input_size=input_size, in_channels=in_channels, num_classes=output_dim)
        else:  # "CNN3"
            model = CNN3(input_size=input_size, in_channels=in_channels, num_classes=output_dim)

    else:
        raise ValueError(f"'{model_type}' not defined as a model_type.")

    return model.to(device)

### Criterion

In [ ]:
class RSB_Loss(nn.Module):
    def __init__(self, alpha=0.01, beta=0.005):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, outputs, targets, model):
        data_loss = F.nll_loss(outputs, targets)

        weights = [p for name, p in model.named_parameters() if "weight" in name]
        norms   = [torch.norm(w)**2 for w in weights]

        if len(norms) < 1:
            return data_loss

        l2_loss = self.alpha * norms[-1]
        balance_terms = [(norms[i] - norms[i+1])**2 for i in range(len(norms) - 1)]
        balance_loss  = self.beta * torch.stack(balance_terms).sum() if balance_terms else norms[-1].new_zeros(1)
        regr_loss     = l2_loss + balance_loss

        return data_loss + regr_loss

In [ ]:
class UNREG_Loss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, outputs, targets, model=None):
        return F.nll_loss(outputs, targets)

In [ ]:
class L2_Loss(nn.Module):
    def __init__(self, alpha: float):
        super().__init__()
        self.alpha = alpha

    def forward(self, outputs, targets, model):
        data_loss = F.nll_loss(outputs, targets)
        l2_loss   = 0.0
        for parameter in model.parameters():
            if parameter.requires_grad:
                l2_loss = l2_loss + torch.sum(parameter.pow(2))
        return data_loss + self.alpha * l2_loss

In [ ]:
def get_criterion(config):
    loss_type = getattr(config, "loss_type", "rsb")

    if loss_type == "adamw" or loss_type == "unregularized":
        return UNREG_Loss()

    if loss_type == "rsb":
        return RSB_Loss(alpha=config.alpha, beta=config.beta)

    if loss_type == "l2":
        return L2_Loss(alpha=config.alpha)

    raise ValueError(f"'{loss_type}' not defined as a loss_type.")

### Optimizer

In [ ]:
def get_optimizer(
    optimizer_name: str,
    params,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    momentum: float = 0.9,
    alpha: float = 0.1,
    beta: float = 0.1
):
    match optimizer_name:
        case "adamw":
            return torch.optim.AdamW(
                params,
                lr=lr,
                weight_decay=weight_decay,
            )

        case "sgd":
            return torch.optim.SGD(
                params,
                lr=lr,
                weight_decay=weight_decay,
                momentum=momentum,
            )

        case _:
            raise ValueError(f"'{optimizer_name}' not defined as an optimizer. ")

### Logging Helpers

In [ ]:
def get_layer_norms(model):
    with torch.no_grad():
        weights         = [p for p in model.parameters() if len(p.shape) > 1]
        total_magnitude = sum(torch.sum(torch.abs(w)).item() for w in weights)
    return total_magnitude

In [ ]:
def get_network_balance(model, per_layer = False):
    with torch.no_grad():
        weights = [p for name, p in model.named_parameters() if "weight" in name]
        norms   = [torch.sum(w**2) for w in weights]

        if len(norms) < 2:
            if per_layer:
                return []
            return 0.0

        diffs   = [(norms[i] - norms[i+1])**2 for i in range(len(norms) - 1)]
        balance = torch.stack(diffs)
        
        if per_layer:
            return balance.tolist()
        else:
            return torch.mean(balance).item()

In [ ]:
def check_layer_ordering():
    fnn = FNN3(input_dim=32 * 32 * 3, output_dim=10)
    cnn = CNN3(input_size=(32, 32), in_channels=3, num_classes=10)

    print("FNN3 weight parameters:")
    for name, param in fnn.named_parameters():
        if "weight" in name:
            print(name)

    print("\nCNN3 weight parameters:")
    for name, param in cnn.named_parameters():
        if "weight" in name:
            print(name)

### Balancing functions (Unfinished)

In [ ]:
# # Baldi's Partial Balancing 
# def stochastic_balance(self):
#     # print("stochastic_balance start")
#     with torch.no_grad() :
#         iii = random.randint(0, 4)

#         if iii == 0:
#             norm_in = (self.fc1.weight.data**2).sum(dim=1)
#             norm_out =  (self.fc2.weight.data**2).sum(dim=0)
#             ratio = (norm_out/norm_in ).sqrt().sqrt()

#             balance_neurons_in_the_middle_of_fc_layers(self.fc1, self.fc2, ratio = ratio, n = 999999999999)

#         if iii == 1:
#             norm_in = (self.fc2.weight.data**2).sum(dim=1)
#             norm_out =  (self.fc3.weight.data**2).sum(dim=0)
#             ratio = (norm_out/norm_in ).sqrt().sqrt()

#             balance_neurons_in_the_middle_of_fc_layers(self.fc2, self.fc3, ratio = ratio, n = 999999999999)

#         if iii == 2:
#             norm_in = (self.fc3.weight.data**2).sum(dim=1)
#             norm_out =  (self.fc4.weight.data**2).sum(dim=0)
#             ratio = (norm_out/norm_in ).sqrt().sqrt()

#             balance_neurons_in_the_middle_of_fc_layers(self.fc3, self.fc4, ratio = ratio, n = 999999999999)

#         if iii == 3:
#             norm_in = (self.fc4.weight.data**2).sum(dim=1)
#             norm_out =  (self.fc5.weight.data**2).sum(dim=0)
#             ratio = (norm_out/norm_in ).sqrt().sqrt()

#             balance_neurons_in_the_middle_of_fc_layers(self.fc4, self.fc5, ratio = ratio, n = 999999999999)

#         if iii == 4:
#             norm_in = (self.fc5.weight.data**2).sum(dim=1)
#             norm_out =  (self.fc6.weight.data**2).sum(dim=0)
#             ratio = (norm_out/norm_in ).sqrt().sqrt()
            
#             balance_neurons_in_the_middle_of_fc_layers(self.fc5, self.fc6, ratio = ratio, n = 99999999999)

In [ ]:
# # Between-layer balancing opertion
# def balance_neurons_in_the_middle_of_fc_layers(fc1, fc2, ratio, n):
#     with torch.no_grad() :
#         nn=0
        
#         for i in random.sample(range(ratio.shape[0]), ratio.shape[0]):
#             fc1.weight.data [i,:] *= ratio[  i ].item()
#             fc2.weight.data [:,i] /= ratio[  i ].item()

#             nn += 1
                
#             if nn > n:
#                 break

### Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[device]: ", device)

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    is_cnn = isinstance(model, (CNN1, CNN2, CNN3))
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            if not is_cnn:
                data = data.view(data.size(0), -1)
            target = target.to(device)

            outputs = model(data)
            loss = criterion(outputs, target, model)
            total_loss = total_loss + loss.item()

            predictions = outputs.argmax(dim=1)
            correct = correct + (predictions == target).sum().item()
            total = total + target.size(0)

    average_loss = total_loss / max(len(data_loader), 1)
    accuracy = correct / max(total, 1)

    return average_loss, accuracy

In [ ]:
def run_experiment():
    run = wandb.init(project=PROJECT_NAME, config=default_config, resume="allow")
    config = wandb.config
    wandb.run.name = (
        f"opt={config.optimizer}_"
        f"model={config.model_type}_"
        f"data={config.dataset}_"
        f"lr={config.lr:.0e}_"
        f"bs={config.batch_size}_"
        f"a={config.alpha}_b={config.beta}"
    )
    train(wandb.config)

def train(config):
    model = get_model(config, device)
    criterion = get_criterion(config)
    train_loader, val_loader, test_loader = get_data_loaders(
        dataset_type=config.dataset,
        batch_size=config.batch_size,
        val_fraction=getattr(config, "val_fraction", 0.1),
        test_fraction=0.2,
    )
    optimizer = get_optimizer(
        optimizer_name=config.optimizer,
        params=model.parameters(),
        lr=config.lr,
        weight_decay=getattr(config, "weight_decay", 0.0),
        momentum=getattr(config, "momentum", 0.9),
        alpha=config.alpha,
        beta=config.beta,
    )

    is_cnn = isinstance(model, (CNN1, CNN2, CNN3))
    step   = 0
    for epoch in range(config.epochs):
        model.train()
        running_loss = 0.0
        correct      = 0
        total        = 0
        log_interval = max(1, len(train_loader) // 30)

        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.to(device)
            if not is_cnn:
                data = data.view(data.size(0), -1)
            target = target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss   = criterion(output, target, model)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            pred = output.argmax(dim=1)

            correct += (pred == target).sum().item()
            total   += target.size(0)
            step    += 1

            if (batch_idx + 1) % log_interval == 0:
                batch_loss = running_loss / (batch_idx + 1)
                batch_acc  = correct / total

                val_loss, val_acc = evaluate(model, val_loader, criterion, device)

                per_layer_balance = get_network_balance(model, per_layer=True)

                log_dict = {
                    "step": step,
                    "epoch": epoch + (batch_idx + 1) / len(train_loader),
                    "train_loss": batch_loss,
                    "train_accuracy": batch_acc,
                    "val_loss": val_loss,
                    "val_accuracy": val_acc,
                    "weight_magnitude": get_layer_norms(model),
                }
                for i, b in enumerate(per_layer_balance):
                    log_dict[f"balance/layer_{i}"] = b
                wandb.log(log_dict, step=step)

        avg_loss = running_loss / len(train_loader)
        avg_acc  = correct / total
        
        val_loss, val_acc   = evaluate(model, val_loader, criterion, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        balance           = get_network_balance(model)
        per_layer_balance = get_network_balance(model, per_layer=True)
        
        log_dict = {
            "epoch": epoch,
            "train_loss": avg_loss,
            "train_accuracy": avg_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "network_balance": balance,
        }
        for i, b in enumerate(per_layer_balance):
            log_dict[f"balance/layer_{i}"] = b
        wandb.log(log_dict, step=step)

In [ ]:
SWEEP_ID_FILE = ".wandb_sweep_id"

def run_sweep():
    if os.path.exists(SWEEP_ID_FILE):
        with open(SWEEP_ID_FILE) as f:
            sweep_id = f.read().strip()
    else:
        sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
        with open(SWEEP_ID_FILE, "w") as f:
            f.write(sweep_id)
    wandb.agent(sweep_id, function=run_experiment)

In [ ]:
wandb.login(key=os.environ.get("WANDB_KEY"))

In [ ]:
run_sweep()